# Re-Verification — Reproductive Entities (Ri, N_R_i, F_R_i)

Runs `prompt_verification_reprod_final.md` on every JSON file in `dataset_ds_reverif/`.  
- **Model**: `deepseek-v4-pro` (thinking mode) — strict logical audit, deterministic.  
- **Input fields sent to the model**: `id`, `specie`, `specie_en`, `corrected_text`  
- **Result saved**: key `verification_reprod` added to each file (skipped if already present)


In [ ]:
import json
import os
import time
import urllib.request
import urllib.error
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("__file__").resolve().parents[2] / ".env") if "__file__" in dir() else load_dotenv()

# ── Configuration ─────────────────────────────────────────────────────────────
API_KEY             = os.environ.get("DEEPSEEK_API_KEY")

MODEL               = "deepseek-v4-pro"
API_URL             = "https://api.deepseek.com/chat/completions"
INPUT_DIR           = Path("../reprod_errors_by_batch/batch_01")
PROMPT_FILE         = Path("../verif_prompts/prompt_verification_reprod_new.md")

# ── Generation parameters ─────────────────────────────────────────────────────
# temperature 0.0      → deterministic, reproducible audit across the full run.
# response_format json → forces strict JSON output, removes ```json stripping.
# max_tokens high      → corrected_text_final can be long; avoid truncation.
TEMPERATURE         = 0.0
MAX_TOKENS          = 8192
USE_JSON_MODE       = True

# ── Rate-limit / retry handling ───────────────────────────────────────────────
DELAY_BETWEEN_CALLS = 5
MAX_RETRIES         = 6
RETRY_DELAY         = 30
# ──────────────────────────────────────────────────────────────────────────────

if not API_KEY:
    raise ValueError("DEEPSEEK_API_KEY environment variable is not set.")

SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")

json_files = sorted(INPUT_DIR.glob("*.json"))
print(f"Model            : {MODEL}")
print(f"API URL          : {API_URL}")
print(f"Temperature      : {TEMPERATURE}")
print(f"JSON mode        : {USE_JSON_MODE}")
print(f"Prompt file      : {PROMPT_FILE.name} ({len(SYSTEM_PROMPT):,} chars)")
print(f"Found            : {len(json_files)} file(s) in {INPUT_DIR}")


Model            : deepseek-v4-pro
API URL          : https://api.deepseek.com/chat/completions
Temperature      : 0.0
JSON mode        : True
Prompt file      : prompt_verification_reprod_new.md (27,749 chars)
Found            : 259 file(s) in ../reprod_errors_by_batch/batch_01
Already verified : 0 / 259


In [2]:
def call_deepseek(system_prompt: str, user_content: str) -> tuple[str, dict]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]
    payload = {
        "model":       MODEL,
        "messages":    messages,
        "temperature": TEMPERATURE,
        "max_tokens":  MAX_TOKENS,
    }
    if USE_JSON_MODE:
        payload["response_format"] = {"type": "json_object"}
    body = json.dumps(payload).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=300) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"], data.get("usage", {})


# ── Main loop ─────────────────────────────────────────────────────────────────
fatal = False
for idx, filepath in enumerate(json_files, 1):
    data = json.loads(filepath.read_text(encoding="utf-8"))

    if data.get("verification_reprod") is not None:
        print(f"[SKIP-verified]  {filepath.name}")
        continue

    if not data.get("corrected_text"):
        print(f"[SKIP-no-text]   {filepath.name}")
        continue

    # Build user payload — specie_en may be absent in these files
    user_payload = json.dumps({
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "specie_en":      data.get("specie_en"),   # null if not present
        "corrected_text": data.get("corrected_text"),
    }, ensure_ascii=False)

    print(f"[{idx:04d}/{len(json_files)}] {filepath.name} ...", end=" ", flush=True)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw_result, usage = call_deepseek(SYSTEM_PROMPT, user_payload)

            try:
                verification = json.loads(raw_result)
            except json.JSONDecodeError:
                stripped = raw_result.strip().removeprefix("```json").removesuffix("```").strip()
                verification = json.loads(stripped)

            data["verification_reprod"] = verification
            filepath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

            verdict   = verification.get("verdict", "?")
            critical  = verification.get("critical_errors", "?")
            cached    = usage.get("prompt_cache_hit_tokens", 0)
            print(f"{verdict} | {critical} critical | {usage.get('prompt_tokens','?')} in / {usage.get('completion_tokens','?')} out / {cached} cached")
            break

        except urllib.error.HTTPError as e:
            body_err = e.read().decode(errors="replace")
            print(f"\n  HTTP {e.code} on attempt {attempt}: {body_err[:200]}")
            if e.code == 401:
                print("  Invalid API key — check DEEPSEEK_API_KEY. Aborting.")
                fatal = True
                break
            elif e.code == 404:
                print(f"  Model '{MODEL}' not found. Aborting.")
                fatal = True
                break
            elif e.code == 403:
                print("  Access forbidden — skipping this file.")
                break
            elif e.code == 429:
                wait = RETRY_DELAY * (2 ** (attempt - 1))
                print(f"  Rate limited — waiting {wait}s before retry...", flush=True)
                time.sleep(wait)
            elif attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
        except Exception as e:
            print(f"\n  Error on attempt {attempt}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    else:
        print(f"  FAILED after {MAX_RETRIES} attempts — skipping.")

    if fatal:
        break

    if idx < len(json_files):
        time.sleep(DELAY_BETWEEN_CALLS)

print("\nAll done.")


[0001/259] 0003_Ablennes_hians_abj.json ... FAIL | 1 critical | 8219 in / 2102 out / 8192 cached
[0002/259] 0008_Abramis_brama_abj.json ... FAIL | 3 critical | 8293 in / 3402 out / 8192 cached
[0003/259] 0009_Abroscopus_superciliaris_std.json ... FAIL | 2 critical | 8109 in / 7680 out / 8064 cached
[0004/259] 0010_Abrothrix_andinus_stx.json ... FAIL | 4 critical | 8312 in / 7147 out / 8192 cached
[0005/259] 0012_Abrothrix_longipilis_stx.json ... FAIL | 1 critical | 8191 in / 2818 out / 7040 cached
[0006/259] 0013_Abrothrix_olivaceus_stx.json ... FAIL | 1 critical | 8201 in / 5009 out / 7040 cached
[0007/259] 0016_Acantharchus_pomotis_abj.json ... FAIL | 4 critical | 8227 in / 7378 out / 7040 cached
[0008/259] 0018_Acanthis_flammea_std.json ... FAIL | 2 critical | 8266 in / 2460 out / 7040 cached
[0009/259] 0019_Acanthis_hornemanni_std.json ... FAIL | 2 critical | 8197 in / 4297 out / 7040 cached
[0010/259] 0020_Acanthisitta_chloris_std.json ... FAIL | 2 critical | 8351 in / 2974 out / 

KeyboardInterrupt: 

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
from collections import Counter

counts = Counter()
for filepath in sorted(INPUT_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    if not data.get("corrected_text"):
        counts["NO_TEXT"] += 1
        continue
    v = data.get("verification_reprod")
    if v is None:
        counts["NOT_VERIFIED"] += 1
    else:
        counts[v.get("verdict", "UNKNOWN")] += 1

total = sum(counts.values())
print(f"=== Re-Verification Summary ({total} files) ===")
for verdict, count in counts.most_common():
    print(f"  {verdict}: {count}")

In [ ]:
# ── Detail: FAIL files ────────────────────────────────────────────────────────
fails = []
for filepath in sorted(INPUT_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification_reprod")
    if v and v.get("verdict") == "FAIL":
        fails.append({
            "file":            filepath.name,
            "specie":          data.get("specie"),
            "critical_errors": v.get("critical_errors"),
            "error_codes":     [e.get("code") for e in v.get("errors", [])],
        })

print(f"FAIL ({len(fails)} files):")
for r in fails:
    print(f"  {r['file']}  —  {r['specie']}  —  {r['critical_errors']} critical  —  {r['error_codes']}")

In [ ]:
# ── Detail: GENERATION_FAILURE files ─────────────────────────────────────────
failures = []
for filepath in sorted(INPUT_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification_reprod")
    if v and v.get("verdict") == "GENERATION_FAILURE":
        failures.append({"file": filepath.name, "specie": data.get("specie")})

print(f"GENERATION_FAILURE ({len(failures)} files):")
for r in failures:
    print(f"  {r['file']}  —  {r['specie']}")